In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xg
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
fPath = '/home/killua/Desktop/files/ml/data/train.csv'
data = pd.read_csv(fPath)
# data.info()
# print(data.isnull().sum())
df = pd.DataFrame(data)
df['Postal Code'].fillna(42420, inplace = True)
print(df['Postal Code'].isnull().sum())



In [26]:
# print(df.isnull().sum())
# df.describe()

df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
sales_by_date = df.groupby('Order Date')['Sales'].sum().reset_index()
sales_by_date['DayOfWeek'] = sales_by_date['Order Date'].dt.dayofweek
sales_by_date['Month'] = sales_by_date['Order Date'].dt.month
sales_by_date['Year'] = sales_by_date['Order Date'].dt.year
sales_by_date.info()

# fig = px.line(sales_by_date, x="Order Date", y="Sales", title='Sales By Date')
# fig.show()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Order Date  1230 non-null   datetime64[ns]
 1   Sales       1230 non-null   float64       
 2   DayOfWeek   1230 non-null   int32         
 3   Month       1230 non-null   int32         
 4   Year        1230 non-null   int32         
dtypes: datetime64[ns](1), float64(1), int32(3)
memory usage: 33.8 KB


In [27]:
def lagged_features(df, lag):
    lagged_data = df.copy()
    for i in range(1, lag+1):
        lagged_data[f'lag_{i}'] = lagged_data['Sales'].shift(i)
    return lagged_data

lag_sales = lagged_features(df[['Order Date','Sales']], 20)

lag_sales = lag_sales.dropna()
# print(lag_sales)

In [28]:
X = lag_sales.drop(columns=['Order Date', 'Sales'])
y = lag_sales['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model_xgb = xg.XGBRegressor(
    objective = 'reg:squarederror', 
    n_estimators = 100, 
    learning_rate=0.1, 
    max_depth = 7,
    random_state = 42
    )
model_xgb.fit(X_train, y_train)

prediction_xgb = model_xgb.predict(X_test)
rmse_xgb = np.sqrt(mean_squared_error(y_test, prediction_xgb))

print(f"RMSE : {rmse_xgb:.2f}")


RMSE : 700.50


In [29]:
X_test_dates = lag_sales.loc[X_test.index]['Order Date']

fig = go.Figure()

# Trace 1: Actual full historical sales
fig.add_trace(go.Scatter(
    x=sales_by_date['Order Date'], 
    y=sales_by_date['Sales'],
    mode='lines', 
    line=dict(color='blue'),
    name='Actual Sales History'
))

# Trace 2: Predicted values plotted only on the test set dates
fig.add_trace(go.Scatter(
    x=X_test_dates,                 # Use the corresponding test dates
    y=prediction_xgb,               # The array of predictions
    mode='lines', 
    line=dict(color='green', dash='dot'),
    name='Predicted Sales (Test Period)'
))

fig.update_layout(title='Prediction vs Actual',
                  xaxis_title='Date',
                  yaxis_title='Sales')

# Show the plot
fig.show()